# Online Retail — Customer Segment Classification
**Dataset:** UCI Online Retail Dataset  
**Goal:** Classify customers into Low / Medium / High spenders based on past behavior  
**Models:** Logistic Regression vs SVM (7 features vs 13 features)

### Design
- Features built from **past transactions (before Jun 2011)**
- Target: which spending segment the customer falls into **after Jun 2011**
- Segments defined by equal thirds (33/33/33) of future spend

---

## 0. Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

---
## 1. Load & Clean Data

In [ ]:
df = pd.read_excel('Online Retail.xlsx', dtype={'CustomerID': str})

df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df.dropna(subset=['CustomerID', 'Description'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['Revenue']     = df['Quantity'] * df['UnitPrice']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f'Clean dataset: {df.shape[0]:,} rows')
df.head()

---
## 2. Temporal Split

Features from **before Jun 2011** → Target from **after Jun 2011**

In [ ]:
cutoff_date = pd.Timestamp('2011-06-01')
past   = df[df['InvoiceDate'] <  cutoff_date].copy()
future = df[df['InvoiceDate'] >= cutoff_date].copy()

print(f'Past transactions   : {len(past):,}')
print(f'Future transactions : {len(future):,}')

---
## 3. Feature Engineering

### Core RFM + order pattern features (7)

In [ ]:
# Core RFM features
past_features = past.groupby('CustomerID').agg(
    Recency         = ('InvoiceDate', lambda x: (cutoff_date - x.max()).days),
    Frequency       = ('InvoiceNo',   'nunique'),
    Monetary_Past   = ('Revenue',     'sum'),
    Unique_Products = ('StockCode',   'nunique'),
    Max_Order       = ('Revenue',     'max'),
    Std_Order       = ('Revenue',     'std'),
    Avg_Items       = ('Quantity',    'mean'),
).reset_index()
past_features['Std_Order'] = past_features['Std_Order'].fillna(0)

print('Core features built.')
past_features.head()

### Behavioral features (6 additional)

In [ ]:
# Tenure — days between first and last purchase
tenure = past.groupby('CustomerID').agg(
    First_Purchase=('InvoiceDate','min'),
    Last_Purchase =('InvoiceDate','max')
).reset_index()
tenure['Tenure'] = (tenure['Last_Purchase'] - tenure['First_Purchase']).dt.days
tenure = tenure[['CustomerID','Tenure']]

# Weekend & Morning shopping patterns
past['Is_Weekend'] = past['InvoiceDate'].dt.dayofweek >= 5
past['Is_Morning'] = past['InvoiceDate'].dt.hour < 12
time_feats = past.groupby('CustomerID').agg(
    Pct_Weekend=('Is_Weekend','mean'),
    Pct_Morning=('Is_Morning','mean')
).reset_index()

# Product category diversity
# StockCode first 2 chars = rough product category
past['Category'] = past['StockCode'].astype(str).str[:2]
categories = past.groupby('CustomerID').agg(
    Unique_Categories=('Category','nunique'),
    Top_Category_Pct =('Category', lambda x: x.value_counts().iloc[0]/len(x))
).reset_index()

# Active weeks — how regularly they shop
past['Week'] = past['InvoiceDate'].dt.isocalendar().week.astype(int)
active_weeks = past.groupby('CustomerID').agg(
    Active_Weeks=('Week','nunique')
).reset_index()

print('Behavioral features built.')

### Merge + Log Transform

In [ ]:
# Future target
future_target = (future.groupby('CustomerID')['Revenue']
                       .sum().reset_index()
                       .rename(columns={'Revenue':'Future_Monetary'}))

# Merge everything
model_df = past_features.copy()
for fdf in [tenure, time_feats, categories, active_weeks, future_target]:
    model_df = model_df.merge(fdf, on='CustomerID', how='left')

model_df = model_df[model_df['Future_Monetary'] > 0].copy()

# Fill nulls
for col in ['Tenure','Pct_Weekend','Pct_Morning','Unique_Categories','Active_Weeks']:
    model_df[col] = model_df[col].fillna(0)
model_df['Top_Category_Pct'] = model_df['Top_Category_Pct'].fillna(1)

# Log transform skewed features
for col in ['Recency','Frequency','Monetary_Past','Unique_Products',
            'Max_Order','Std_Order','Avg_Items','Future_Monetary',
            'Tenure','Unique_Categories','Active_Weeks']:
    model_df[f'Log_{col}'] = np.log1p(model_df[col])

print(f'Final dataset: {model_df.shape[0]:,} customers, {model_df.shape[1]} columns')
model_df.head()

---
## 4. Create Target Segments

Split customers into 3 equal groups based on future spend:

In [ ]:
model_df['Segment'] = pd.qcut(
    model_df['Future_Monetary'],
    q=3,
    labels=[0, 1, 2]   # 0=Low, 1=Medium, 2=High
).astype(int)

print('--- Segment Distribution ---')
counts = model_df['Segment'].value_counts().sort_index()
for seg, label in zip([0,1,2], ['Low','Medium','High']):
    print(f'  {label:8} (class {seg}): {counts[seg]:,} customers '
          f'({counts[seg]/len(model_df)*100:.1f}%)')

print('\n--- Spend Ranges per Segment ---')
print(model_df.groupby('Segment')['Future_Monetary']
              .agg(['min','max','mean']).round(2))

In [ ]:
# Visualise segment distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, (col, title) in enumerate([
    ('Future_Monetary', 'Future Spend by Segment'),
    ('Frequency',       'Frequency by Segment'),
    ('Recency',         'Recency by Segment'),
]):
    for seg, lbl in zip([0,1,2], ['Low','Medium','High']):
        axes[i].hist(model_df[model_df['Segment']==seg][col],
                     bins=40, alpha=0.5, label=lbl, edgecolor='white')
    axes[i].set_title(title)
    axes[i].set_xlabel(col)
    axes[i].legend()
plt.suptitle('Feature Distributions by Segment', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Correlation Analysis

In [ ]:
features_13 = [
    'Log_Recency','Log_Frequency','Log_Monetary_Past',
    'Log_Unique_Products','Log_Max_Order','Log_Std_Order','Log_Avg_Items',
    'Log_Tenure','Pct_Weekend','Pct_Morning',
    'Log_Unique_Categories','Top_Category_Pct','Log_Active_Weeks'
]

corr_df = model_df[features_13 + ['Segment']].corr()[['Segment']].drop('Segment')
corr_df = corr_df.sort_values('Segment', ascending=False)

plt.figure(figsize=(5, 8))
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5)
plt.title('Feature Correlation with Segment')
plt.tight_layout()
plt.show()

---
## 6. Model Training

Four models compared:
1. Logistic Regression — 7 core features
2. Logistic Regression — 13 enriched features
3. SVM (RBF kernel) — 7 core features
4. SVM (RBF kernel) — 13 enriched features

> **Note:** Both LR and SVM are sensitive to feature scale — StandardScaler applied before training.

In [ ]:
features_7  = [
    'Log_Recency','Log_Frequency','Log_Monetary_Past',
    'Log_Unique_Products','Log_Max_Order','Log_Std_Order','Log_Avg_Items'
]
features_13 = features_7 + [
    'Log_Tenure','Pct_Weekend','Pct_Morning',
    'Log_Unique_Categories','Top_Category_Pct','Log_Active_Weeks'
]

y = model_df['Segment']
all_results = []
all_models  = {}

for label, feats, clf in [
    ('LR — 7ft',   features_7,  LogisticRegression(max_iter=1000, random_state=42)),
    ('LR — 13ft',  features_13, LogisticRegression(max_iter=1000, random_state=42)),
    ('SVM — 7ft',  features_7,  SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)),
    ('SVM — 13ft', features_13, SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)),
]:
    X = model_df[feats]
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    scaler = StandardScaler()
    clf.fit(scaler.fit_transform(Xtr), ytr)
    yp = clf.predict(scaler.transform(Xte))

    all_results.append({
        'Model'    : label,
        'Accuracy' : accuracy_score(yte, yp),
        'F1'       : f1_score(yte, yp, average='weighted'),
        'Precision': precision_score(yte, yp, average='weighted'),
        'Recall'   : recall_score(yte, yp, average='weighted')
    })
    all_models[label] = (yte, yp)

    print(f'\n--- {label} ---')
    print(classification_report(yte, yp, target_names=['Low','Medium','High']))

---
## 7. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
axes = axes.flatten()

for i, (label, (yte, yp)) in enumerate(all_models.items()):
    cm   = confusion_matrix(yte, yp)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Low','Medium','High'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(f'{label}  |  Accuracy: {accuracy_score(yte,yp):.4f}')

plt.suptitle('Confusion Matrices — All Models', fontsize=13)
plt.tight_layout()
plt.show()

---
## 8. Full Metrics Comparison

In [ ]:
results_df = pd.DataFrame(all_results)
print(results_df.to_string(index=False))

x, width = np.arange(4), 0.2
fig, ax  = plt.subplots(figsize=(13, 6))

for i, (metric, color) in enumerate(zip(
    ['Accuracy','F1','Precision','Recall'],
    ['steelblue','coral','mediumseagreen','mediumpurple']
)):
    bars = ax.bar(x + i*width, results_df[metric], width,
                  label=metric, color=color, edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{bar.get_height():.3f}',
                ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(results_df['Model'], fontsize=10)
ax.set_ylim(0, 0.85)
ax.axhline(0.6, color='black', linestyle='--', linewidth=1, label='0.60 reference')
ax.set_title('All Models — Metrics Comparison')
ax.set_ylabel('Score')
ax.legend()
plt.tight_layout()
plt.show()

---
## 9. Per-class Performance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, metric in enumerate(['precision','recall','f1-score']):
    for j, (label, (yte, yp)) in enumerate(all_models.items()):
        report = classification_report(yte, yp, output_dict=True)
        scores = [report[str(k)][metric] for k in range(3)]
        xpos   = np.arange(3) + j*0.2
        axes[i].bar(xpos, scores, 0.18,
                    label=label,
                    color=['steelblue','coral','mediumseagreen','mediumpurple'][j],
                    edgecolor='white')
    axes[i].set_title(f'{metric.capitalize()} by Segment')
    axes[i].set_xticks(np.arange(3) + 0.3)
    axes[i].set_xticklabels(['Low','Medium','High'])
    axes[i].set_ylim(0, 1.0)
    axes[i].legend(fontsize=7)

plt.suptitle('Per-class Performance — All Models', fontsize=13)
plt.tight_layout()
plt.show()

---
## 10. Key Findings

### Verified results

| Model | Accuracy | F1 |
|---|---|---|
| LR — 7 features | 0.567 | 0.561 |
| LR — 13 features | 0.562 | 0.555 |
| SVM — 7 features | 0.573 | 0.573 |
| SVM — 13 features | **0.583** | **0.581** |

### What the results mean
- Random guessing on 3 classes = 0.33 accuracy. Our best model (0.583) is **77% better than random**
- **Medium segment is hardest to classify** — it borders both Low and High with no sharp boundary
- **High spenders are easiest to identify** — F1 ~0.69–0.70 across all models
- SVM with 13 features is the best model overall

### Why the new features helped SVM but not LR
- `Log_Active_Weeks` (correlation 0.51) and `Log_Tenure` (0.36) added genuine signal
- SVM's RBF kernel can use these non-linearly — LR cannot
- `Pct_Weekend` (-0.07) and `Top_Category_Pct` (-0.06) were noise — barely correlated with segment

### Natural performance ceiling ~0.58
- Only 6 months of past history per customer
- Segment boundaries are soft — £488 (Low max) vs £490 (Medium min) are practically identical
- Customer behavior changes due to life events the model cannot see

### What would improve performance
- Random Forest or XGBoost — handles non-linear segment boundaries better
- More data — 2–3 years of history per customer
- Business-defined segments instead of equal thirds (e.g. top 20% = High)